# 결측 주입과 세 대체법

실제 완료한 paired 실험과 전체 SAML 평가를 나란히 확인합니다.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, HTML
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "saml_d_experiments"]
ROOT = next((p.resolve() for p in candidates if (p / "PACKAGE.json").exists()
             and json.loads((p / "PACKAGE.json").read_text())["package"] == "saml_d_experiments"), None)
if ROOT is None:
    raise FileNotFoundError("압축을 풀고 해당 패키지의 notebooks 폴더에서 실행하세요.")
sys.path.insert(0, str(ROOT / "src"))
print("패키지:", ROOT)


In [ ]:
diagnosis = ROOT/'results/20261106/결측_분포진단_20261106'
controls = pd.read_csv(diagnosis/'control_metrics.csv')
display(controls[['condition','rows','positive','AP','precision','recall','F1','FP']])

HI Test 동일 표본262,144건(양성332)에서 결측 주입. 아래 대체법 Validation은 다른262,144건(양성340)입니다. 표본 Test와 전체 SAML을 같은 모집단으로 해석하지 않습니다.

In [ ]:
stage = ROOT/'results/20261106/대체값_추가실험_20261106'
val = pd.read_csv(stage/'validation_metrics.csv')
saml = pd.read_csv(stage/'saml_metrics.csv')
display(val)
display(saml)

In [ ]:
import matplotlib.pyplot as plt
comparison = val.merge(saml, on='method', suffixes=('_HI_val','_SAML'))
fig, axes = plt.subplots(1,2,figsize=(12,4))
for ax, scope in zip(axes,['HI_val','SAML']):
    ax.bar(comparison.method,100*comparison['AP_'+scope])
    ax.set(title=scope, ylabel='Average Precision (%)')
    ax.tick_params(axis='x',rotation=25)
fig.tight_layout()
plt.show()

HI Val에서 대체가 잘 돼도 SAML 전체에서는 세 방법 모두 낮아졌습니다. 두 패널의 y축이 다릅니다. 결측 민감도는 확인되었으나 대체 방법의 일반화는 실패했습니다. 원본 실행 코드: `reference_source/Multi_FraudGT_SAML_D_전체평가_20261106/대체값_추가실험_20261106/`.